# 03 Train YOLO11n Sign Detector on Roboflow Exports

02 노트북에서 Roboflow ZIP 2개를 검수한 결과, 데이터셋 구조와 대부분의 bbox는 정상임.

다만 아래 작은 문제가 있었음.

- `field2_horn_000001`: train split에서 빈 라벨로 들어감.
- `field2_horn_000456`: valid split에서 bbox가 너무 작은 점 형태로 들어감.

이번 03 노트북은 원본 ZIP은 건드리지 않고, Colab 내부에서 **sanitized dataset**을 만든 뒤 YOLO11n을 학습함.

실험은 두 개로 나눔.

1. `clean_v1`: 증강 없는 baseline
2. `light_aug_v2`: Roboflow light augmentation 적용 버전

둘을 합쳐서 학습하지 않는 이유는 `light_aug_v2`가 이미 clean train 원본을 포함하고 train split만 2배로 늘어난 구조였기 때문임.


## 0. Colab Runtime 준비

Colab에서 실행할 것.

권장 Runtime:

```text
Runtime > Change runtime type > T4 GPU
```

Ultralytics 공식 사용 방식은 아래와 같음.

```python
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data="data.yaml", epochs=100, imgsz=640)
```

이 노트북은 같은 흐름을 우리 Roboflow dataset에 맞춰 구성함.


In [ ]:
!pip -q install ultralytics

from pathlib import Path
import os
import json
import shutil
import zipfile
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

try:
    import ultralytics
    print('ultralytics:', ultralytics.__version__)
except Exception as e:
    print('ultralytics version check failed:', e)


## 1. Google Drive Mount와 경로 설정

아래 셀에서 `DRIVE_ZIP_DIR`만 본인 Drive 위치에 맞게 수정하면 됨.

여기에 Roboflow에서 받은 ZIP 2개를 올려두면 됨.

```text
26-1_Embedded_sign_labeling.v1-field2_sign_640_no_aug.yolov11.zip
26-1_Embedded_sign_labeling.v2-field2_sign_640_light_blur_rotate_noise.yolov11.zip
```

학습 결과는 `DRIVE_OUTPUT_DIR`에 저장함.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# 필요하면 이 경로만 수정하면 됨.
DRIVE_ZIP_DIR = Path('/content/drive/MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/04_YOLO_Sign_Detection/dataset_zips')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/26-1학기_임베디드인공지능시스템최적화/04_YOLO_Sign_Detection/runs_yolo11n')

# Colab 작업 폴더
WORK_ROOT = Path('/content/sign_yolo11n_field2')
RAW_DATA_ROOT = WORK_ROOT / 'raw_roboflow_exports'
SANITIZED_ROOT = WORK_ROOT / 'sanitized_datasets'
REPORT_ROOT = WORK_ROOT / 'reports'

for p in [DRIVE_ZIP_DIR, DRIVE_OUTPUT_DIR, WORK_ROOT, RAW_DATA_ROOT, SANITIZED_ROOT, REPORT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('DRIVE_ZIP_DIR:', DRIVE_ZIP_DIR)
print('DRIVE_OUTPUT_DIR:', DRIVE_OUTPUT_DIR)
print('WORK_ROOT:', WORK_ROOT)


## 2. ZIP 파일 찾기

ZIP 파일 이름으로 clean/aug 버전을 자동 구분함.

만약 여기서 assert가 걸리면, Drive에 ZIP 2개가 없거나 `DRIVE_ZIP_DIR`가 잘못된 것임.


In [ ]:
EXPECTED_CLASSES = ['horn', 'left', 'right', 'speed_20', 'stop', 'straight']
SPLITS = ['train', 'valid', 'test']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

zip_files = sorted(DRIVE_ZIP_DIR.glob('*.zip'))
print('zip files:', len(zip_files))
for z in zip_files:
    print('-', z.name, f'{z.stat().st_size/1024/1024:.2f} MB')

assert len(zip_files) >= 2, f'ZIP 파일 2개가 필요함: {DRIVE_ZIP_DIR}'

def infer_dataset_key(zip_path: Path) -> str:
    name = zip_path.name.lower()
    if 'no_aug' in name or 'clean' in name or 'v1-' in name:
        return 'clean_v1'
    if 'light' in name or 'aug' in name or 'blur' in name or 'rotate' in name or 'noise' in name or 'v2-' in name:
        return 'light_aug_v2'
    return zip_path.stem

zip_records = []
for z in zip_files:
    key = infer_dataset_key(z)
    if key in {'clean_v1', 'light_aug_v2'}:
        zip_records.append({'dataset_key': key, 'zip_path': str(z), 'zip_name': z.name})

zip_df = pd.DataFrame(zip_records).drop_duplicates('dataset_key')
display(zip_df)
assert set(zip_df['dataset_key']) == {'clean_v1', 'light_aug_v2'}, 'clean_v1/light_aug_v2 ZIP을 모두 찾지 못함.'


## 3. ZIP 압축 해제

원본 Roboflow export는 `RAW_DATA_ROOT`에 해제함.

이미 해제되어 있으면 재사용함. 다시 풀고 싶으면 `FORCE_EXTRACT=True`로 바꾸면 됨.


In [ ]:
FORCE_EXTRACT = False

raw_records = []
for _, rec in zip_df.iterrows():
    key = rec['dataset_key']
    zip_path = Path(rec['zip_path'])
    out_dir = RAW_DATA_ROOT / key
    if FORCE_EXTRACT and out_dir.exists():
        shutil.rmtree(out_dir)
    if not (out_dir / 'data.yaml').exists():
        out_dir.mkdir(parents=True, exist_ok=True)
        print('extracting:', zip_path.name, '->', out_dir)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(out_dir)
    else:
        print('reuse:', out_dir)
    raw_records.append({'dataset_key': key, 'raw_dir': str(out_dir)})

raw_df = pd.DataFrame(raw_records)
display(raw_df)


## 4. Sanitized Dataset 생성

02 노트북에서 발견한 문제 파일을 학습 입력에서 제외함.

제외 규칙:

- `train`: `field2_horn_000001_jpg`로 시작하는 이미지/라벨 제거
- `valid`: `field2_horn_000456_jpg`로 시작하는 이미지/라벨 제거

Roboflow가 hash를 붙이기 때문에 파일명 전체가 아니라 base prefix로 제거함.


In [ ]:
EXCLUDE_BASE_BY_SPLIT = {
    'train': ['field2_horn_000001_jpg'],
    'valid': ['field2_horn_000456_jpg'],
    'test': [],
}

# Roboflow가 data.yaml에 ../train/images를 쓰는 경우가 있어, 학습용 yaml은 절대경로로 다시 작성함.
def write_data_yaml(dataset_dir: Path):
    yaml_text = "\n".join([
        f"train: {dataset_dir / 'train' / 'images'}",
        f"val: {dataset_dir / 'valid' / 'images'}",
        f"test: {dataset_dir / 'test' / 'images'}",
        "",
        "nc: 6",
        "names: ['horn', 'left', 'right', 'speed_20', 'stop', 'straight']",
        "",
    ])
    (dataset_dir / 'data.yaml').write_text(yaml_text, encoding='utf-8')
    return dataset_dir / 'data.yaml'

def should_exclude(stem: str, split: str) -> bool:
    return any(stem.startswith(prefix) for prefix in EXCLUDE_BASE_BY_SPLIT.get(split, []))

def sanitize_dataset(src_dir: Path, dst_dir: Path):
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    removed = []
    for split in SPLITS:
        for sub in ['images', 'labels']:
            folder = dst_dir / split / sub
            if not folder.exists():
                continue
            for p in folder.iterdir():
                if p.is_file() and should_exclude(p.stem, split):
                    removed.append({'split': split, 'sub': sub, 'path': str(p), 'stem': p.stem})
                    p.unlink()
    data_yaml = write_data_yaml(dst_dir)
    return removed, data_yaml

sanitized_records = []
removed_rows = []
for _, rec in raw_df.iterrows():
    key = rec['dataset_key']
    src_dir = Path(rec['raw_dir'])
    dst_dir = SANITIZED_ROOT / key
    removed, data_yaml = sanitize_dataset(src_dir, dst_dir)
    for row in removed:
        row['dataset_key'] = key
        removed_rows.append(row)
    sanitized_records.append({'dataset_key': key, 'dataset_dir': str(dst_dir), 'data_yaml': str(data_yaml)})

sanitized_df = pd.DataFrame(sanitized_records)
removed_df = pd.DataFrame(removed_rows)
display(sanitized_df)
display(removed_df)

removed_df.to_csv(REPORT_ROOT / 'sanitized_removed_files.csv', index=False)


## 5. Sanitized Dataset 재검수

학습 직전에 다시 검사함.

확인할 것:

- missing label 0
- orphan label 0
- empty label 0
- label value issue 0


In [ ]:
def collect_split_files(dataset_dir: Path, split: str):
    image_dir = dataset_dir / split / 'images'
    label_dir = dataset_dir / split / 'labels'
    images = sorted([p for p in image_dir.glob('*') if p.suffix.lower() in IMAGE_EXTS]) if image_dir.exists() else []
    labels = sorted(label_dir.glob('*.txt')) if label_dir.exists() else []
    return images, labels, {p.stem: p for p in images}, {p.stem: p for p in labels}

def parse_label(label_path: Path):
    text = label_path.read_text(encoding='utf-8').strip()
    if not text:
        return []
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) != 5:
            rows.append({'parse_ok': False, 'issue': 'not_5_columns', 'raw': line})
            continue
        cls = int(float(parts[0]))
        vals = [float(x) for x in parts[1:]]
        issue = []
        if not (0 <= cls < len(EXPECTED_CLASSES)):
            issue.append('class_id_out_of_range')
        if any(v < 0 or v > 1 for v in vals):
            issue.append('coord_out_of_0_1')
        if vals[2] <= 0 or vals[3] <= 0:
            issue.append('non_positive_wh')
        if vals[2] < 0.005 or vals[3] < 0.005:
            issue.append('very_tiny_box')
        rows.append({
            'parse_ok': True,
            'class_id': cls,
            'class_name': EXPECTED_CLASSES[cls] if 0 <= cls < len(EXPECTED_CLASSES) else 'INVALID',
            'x_center': vals[0], 'y_center': vals[1], 'width': vals[2], 'height': vals[3],
            'issue': ';'.join(issue),
            'raw': line,
        })
    return rows

summary_rows = []
box_rows = []
issue_rows = []
empty_rows = []
missing_rows = []
orphan_rows = []

for _, rec in sanitized_df.iterrows():
    key = rec['dataset_key']
    d = Path(rec['dataset_dir'])
    for split in SPLITS:
        images, labels, image_by_stem, label_by_stem = collect_split_files(d, split)
        missing = sorted(set(image_by_stem) - set(label_by_stem))
        orphan = sorted(set(label_by_stem) - set(image_by_stem))
        empty = [stem for stem, p in label_by_stem.items() if p.stat().st_size == 0]
        summary_rows.append({
            'dataset_key': key, 'split': split,
            'image_count': len(images), 'label_count': len(labels),
            'missing_label_count': len(missing), 'orphan_label_count': len(orphan), 'empty_label_count': len(empty),
        })
        for stem in missing:
            missing_rows.append({'dataset_key': key, 'split': split, 'stem': stem})
        for stem in orphan:
            orphan_rows.append({'dataset_key': key, 'split': split, 'stem': stem})
        for stem in empty:
            empty_rows.append({'dataset_key': key, 'split': split, 'stem': stem})
        for label_path in labels:
            parsed = parse_label(label_path)
            for row in parsed:
                out = {'dataset_key': key, 'split': split, 'stem': label_path.stem, **row}
                box_rows.append(out)
                if row.get('issue'):
                    issue_rows.append(out)

summary_df = pd.DataFrame(summary_rows)
box_df = pd.DataFrame(box_rows)
issue_df = pd.DataFrame(issue_rows)
class_dist_df = box_df.groupby(['dataset_key', 'split', 'class_id', 'class_name']).size().reset_index(name='box_count')

display(summary_df)
display(class_dist_df)
print('issues:', len(issue_df), 'empty:', len(empty_rows), 'missing:', len(missing_rows), 'orphan:', len(orphan_rows))
if len(issue_df):
    display(issue_df)

summary_df.to_csv(REPORT_ROOT / 'sanitized_split_summary.csv', index=False)
class_dist_df.to_csv(REPORT_ROOT / 'sanitized_class_distribution.csv', index=False)
issue_df.to_csv(REPORT_ROOT / 'sanitized_label_issues.csv', index=False)

assert len(issue_df) == 0, 'label issue가 남아 있음.'
assert len(empty_rows) == 0, 'empty label이 남아 있음.'
assert len(missing_rows) == 0, 'missing label이 남아 있음.'
assert len(orphan_rows) == 0, 'orphan label이 남아 있음.'
print('sanitized dataset audit passed')


## 6. Smoke Training Option

Colab 환경/데이터 경로가 정상인지 확인하려면 먼저 3 epoch smoke run을 할 수 있음.

시간이 없으면 `RUN_SMOKE = False`로 두고 바로 full training으로 넘어가도 됨.


In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    smoke_yaml = sanitized_df[sanitized_df['dataset_key'] == 'clean_v1']['data_yaml'].iloc[0]
    smoke_model = YOLO('yolo11n.pt')
    smoke_results = smoke_model.train(
        data=smoke_yaml,
        epochs=3,
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else 'cpu',
        project=str(DRIVE_OUTPUT_DIR),
        name='smoke_clean_yolo11n',
        exist_ok=True,
        seed=0,
        patience=3,
        workers=2,
        cache=False,
    )
else:
    print('RUN_SMOKE=False. smoke training skipped.')


## 7. 학습 설정

추천 시작값:

- model: `yolo11n.pt`
- imgsz: 640
- epochs: 80
- batch: 16
- patience: 20

결과가 애매하면 이후 `epochs=120`이나 `yolo11s.pt`까지 비교할 수 있지만, 첫 실험은 nano 모델로 속도와 정확도의 균형을 확인함.


In [ ]:
RUN_CLEAN_TRAINING = True
RUN_AUG_TRAINING = True

MODEL_NAME = 'yolo11n.pt'
IMGSZ = 640
EPOCHS = 80
BATCH = 16
PATIENCE = 20
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
SEED = 0
WORKERS = 2

TRAIN_CONFIG = {
    'model': MODEL_NAME,
    'imgsz': IMGSZ,
    'epochs': EPOCHS,
    'batch': BATCH,
    'patience': PATIENCE,
    'device': DEVICE,
    'seed': SEED,
    'workers': WORKERS,
    'output_dir': str(DRIVE_OUTPUT_DIR),
}
print(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2))
(REPORT_ROOT / 'train_config.json').write_text(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2), encoding='utf-8')


## 8. clean v1 학습

clean v1은 baseline임.

이 결과가 낮으면 augmentation 문제가 아니라 라벨/모델/데이터 자체 문제를 의심해야 함.  
이 결과가 높고 aug가 더 높으면 augmentation이 도움이 된 것으로 해석할 수 있음.


In [ ]:
train_run_records = []

def train_one(dataset_key: str, run_name: str):
    row = sanitized_df[sanitized_df['dataset_key'] == dataset_key].iloc[0]
    data_yaml = row['data_yaml']
    print('training:', dataset_key)
    print('data:', data_yaml)
    model = YOLO(MODEL_NAME)
    results = model.train(
        data=data_yaml,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(DRIVE_OUTPUT_DIR),
        name=run_name,
        exist_ok=True,
        seed=SEED,
        patience=PATIENCE,
        workers=WORKERS,
        cache=False,
        verbose=True,
    )
    run_dir = Path(results.save_dir)
    best_pt = run_dir / 'weights' / 'best.pt'
    last_pt = run_dir / 'weights' / 'last.pt'
    record = {
        'dataset_key': dataset_key,
        'run_name': run_name,
        'run_dir': str(run_dir),
        'best_pt': str(best_pt),
        'last_pt': str(last_pt),
        'data_yaml': data_yaml,
    }
    train_run_records.append(record)
    print('run_dir:', run_dir)
    print('best_pt exists:', best_pt.exists())
    return record

if RUN_CLEAN_TRAINING:
    clean_record = train_one('clean_v1', 'field2_clean_v1_yolo11n')
else:
    print('RUN_CLEAN_TRAINING=False. skipped.')


## 9. light-aug v2 학습

light-aug v2는 실전 후보임.

조명 변화, 약한 blur/noise/rotation이 들어간 train split으로 학습하므로 실제 map 조명과 약간의 흔들림에 더 강해질 가능성이 있음.


In [ ]:
if RUN_AUG_TRAINING:
    aug_record = train_one('light_aug_v2', 'field2_light_aug_v2_yolo11n')
else:
    print('RUN_AUG_TRAINING=False. skipped.')


## 10. 결과표 정리

Ultralytics가 저장한 `results.csv`에서 마지막 epoch 기준 주요 metric을 모음.

최종 판단은 단순히 마지막 epoch만 보지 말고, Roboflow/Ultralytics가 저장한 confusion matrix와 PR curve도 함께 확인해야 함.


In [ ]:
def summarize_results_csv(run_dir: Path):
    csv_path = run_dir / 'results.csv'
    if not csv_path.exists():
        return {'results_csv': str(csv_path), 'exists': False}
    df = pd.read_csv(csv_path)
    # columns에 공백이 들어가는 경우가 있어 정리
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1].to_dict()
    best_idx = None
    map_cols = [c for c in df.columns if 'mAP50(B)' in c or 'metrics/mAP50(B)' in c]
    if map_cols:
        best_idx = int(df[map_cols[0]].idxmax())
        best = df.iloc[best_idx].to_dict()
    else:
        best = last
    wanted = ['epoch', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'train/box_loss', 'train/cls_loss', 'val/box_loss', 'val/cls_loss']
    out = {'results_csv': str(csv_path), 'exists': True, 'best_epoch_by_mAP50': best_idx}
    for k in wanted:
        if k in best:
            out[f'best_{k}'] = best[k]
        if k in last:
            out[f'last_{k}'] = last[k]
    return out

summary_records = []
for rec in train_run_records:
    run_dir = Path(rec['run_dir'])
    summary = {**rec, **summarize_results_csv(run_dir)}
    summary_records.append(summary)

train_summary_df = pd.DataFrame(summary_records)
display(train_summary_df)
summary_csv = DRIVE_OUTPUT_DIR / 'field2_yolo11n_train_summary.csv'
train_summary_df.to_csv(summary_csv, index=False)
print('saved:', summary_csv)


## 11. Test Split Evaluation

학습 중 validation은 valid split 기준임.  
아래 셀은 각 best model을 test split으로 다시 평가함.


In [ ]:
RUN_TEST_EVAL = True

eval_records = []
if RUN_TEST_EVAL:
    for rec in train_run_records:
        best_pt = Path(rec['best_pt'])
        if not best_pt.exists():
            print('missing best:', best_pt)
            continue
        model = YOLO(str(best_pt))
        metrics = model.val(
            data=rec['data_yaml'],
            split='test',
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            project=str(DRIVE_OUTPUT_DIR),
            name=f"test_eval_{rec['run_name']}",
            exist_ok=True,
        )
        eval_records.append({
            'dataset_key': rec['dataset_key'],
            'run_name': rec['run_name'],
            'best_pt': rec['best_pt'],
            'test_map50': float(metrics.box.map50),
            'test_map50_95': float(metrics.box.map),
            'test_precision_mean': float(np.mean(metrics.box.p)) if hasattr(metrics.box, 'p') else None,
            'test_recall_mean': float(np.mean(metrics.box.r)) if hasattr(metrics.box, 'r') else None,
        })
else:
    print('RUN_TEST_EVAL=False. skipped.')

eval_df = pd.DataFrame(eval_records)
display(eval_df)
eval_csv = DRIVE_OUTPUT_DIR / 'field2_yolo11n_test_eval_summary.csv'
eval_df.to_csv(eval_csv, index=False)
print('saved:', eval_csv)


## 12. Test Image Prediction Preview

각 best model로 test 이미지 일부를 예측해 저장함.

여기서 확인할 것:

- 표지판을 놓치지 않는지
- left/right/straight를 헷갈리지 않는지
- confidence가 지나치게 낮지 않은지
- background나 기둥까지 크게 잡지 않는지


In [ ]:
RUN_PREDICT_PREVIEW = True
PREDICT_CONF = 0.25
PREDICT_SAMPLES = 36

if RUN_PREDICT_PREVIEW:
    for rec in train_run_records:
        best_pt = Path(rec['best_pt'])
        dataset_dir = Path(sanitized_df[sanitized_df['dataset_key'] == rec['dataset_key']]['dataset_dir'].iloc[0])
        test_images = sorted((dataset_dir / 'test' / 'images').glob('*.jpg'))[:PREDICT_SAMPLES]
        print(rec['run_name'], 'sample images:', len(test_images))
        if not best_pt.exists() or not test_images:
            continue
        model = YOLO(str(best_pt))
        model.predict(
            source=[str(p) for p in test_images],
            imgsz=IMGSZ,
            conf=PREDICT_CONF,
            device=DEVICE,
            save=True,
            project=str(DRIVE_OUTPUT_DIR),
            name=f"predict_test_samples_{rec['run_name']}",
            exist_ok=True,
        )
else:
    print('RUN_PREDICT_PREVIEW=False. skipped.')


## 13. Optional ONNX Export

아직 최종 모델을 고르기 전이라 기본값은 `False`로 둠.

clean/aug 결과를 비교해서 후보를 정한 뒤 export해도 됨.  
당장 Pi 속도 테스트까지 이어가고 싶으면 `EXPORT_ONNX=True`로 바꿔 실행함.


In [ ]:
EXPORT_ONNX = False

if EXPORT_ONNX:
    export_records = []
    for rec in train_run_records:
        best_pt = Path(rec['best_pt'])
        if not best_pt.exists():
            continue
        model = YOLO(str(best_pt))
        export_path = model.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True, dynamic=False)
        export_records.append({
            'dataset_key': rec['dataset_key'],
            'run_name': rec['run_name'],
            'best_pt': rec['best_pt'],
            'onnx_path': str(export_path),
        })
    export_df = pd.DataFrame(export_records)
    display(export_df)
    export_df.to_csv(DRIVE_OUTPUT_DIR / 'field2_yolo11n_onnx_exports.csv', index=False)
else:
    print('EXPORT_ONNX=False. ONNX export skipped.')


## 14. 실행 결과 메모

실행 후 확인할 산출물:

```text
DRIVE_OUTPUT_DIR/
  field2_clean_v1_yolo11n/
    weights/best.pt
    results.csv
    confusion_matrix.png
  field2_light_aug_v2_yolo11n/
    weights/best.pt
    results.csv
    confusion_matrix.png
  field2_yolo11n_train_summary.csv
  field2_yolo11n_test_eval_summary.csv
```

다음 노트북에서는 clean vs light-aug 결과를 비교하고, 최종 Pi 탑재 후보를 고름.
